### CineBot: A movie ticket Booking Assistant

### Tools
Tools are just methods with proper defined input and output and description


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from typing import Literal

load_dotenv()

True

In [2]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
model = ChatOpenAI(model="gpt-5-nano")

In [ ]:
class MovieShows(BaseModel):
  name : str
  timing:str

In [ ]:
struct_model = model.with_structured_output(MovieShows)
response = struct_model.invoke("Is Interstellar showing tonight at 7pm at the Downtown cinema ?")

In [5]:
response

MovieShows(name='Interstellar', timing='Unknown')

<img src="../../assets/tools_definition.png" width="800" height="300">
Tool ->  Args with type hints.

Tools are just glorified Functions/API Calls

* Description in @tool decorator parameter overrides the docstring description.
* Langchain provides a way to define docstring and tool description separately, but is not provided in @tool, it will use docstring as desciption

In [ ]:
@tool
def check_showtimes(movie_title:str) -> str:
  """Check available showtimes for a movie at the cinema.

  Args:
      movie_title: The exact title of the movie to check
  """
  fake_showtimes = {
      "interstellar": "7:00 PM and 10:15 PM",
      "dune part two": "9:30 PM only",
      "oppenheimer": "Sold out for tonight",
  }
  return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

@tool('book_seats', description = 'Book Cinema for a customer, use whenever customer wants to book/reserve a seat.')
def reserve(movie:str,seats:int) ->str:
  """Reserve Seats"""
  return f"Reserved {seats} seat for {movie}"


Arguments Schema
> Reserved argument names<br>
> Never use runtime and config as parameter of your tool, these are reserved by Langchain.
>The following parameter names are reserved and cannot be used as tool arguments. Using these names will cause runtime errors.
>Parameter name	Purpose config Reserved for passing RunnableConfig to tools internally
>runtime Reserved for ToolRuntime parameter (accessing state, context, store)
>To access runtime information, use the ToolRuntime parameter instead of naming your own arguments config or runtime.


In [13]:
class SeatBookingIput(BaseModel):
    movie_title:str = Field(description='Exact Event  Title')
    seat_count : int = Field(description='Number of seats to book', ge=1, le=10)
    preferred_row : Literal['front', 'middle', 'back'] = Field(default='middle', description='Preferred seat row')

@tool(args_schema=SeatBookingIput)
def book_seats(movie_title:str, seats:int, preferred_row:str, config:str)-> str:
    """Book Seats for a Movie"""
    return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

print(book_seats.args)

{'movie_title': {'description': 'Exact Event  Title', 'title': 'Movie Title', 'type': 'string'}, 'seat_count': {'description': 'Number of seats to book', 'maximum': 10, 'minimum': 1, 'title': 'Seat Count', 'type': 'integer'}, 'preferred_row': {'default': 'middle', 'description': 'Preferred seat row', 'enum': ['front', 'middle', 'back'], 'title': 'Preferred Row', 'type': 'string'}}


### Binding VS Execution

In [15]:
model_with_tools = model.bind_tools([check_showtimes, book_seats])
response = model_with_tools.invoke("Is Interstellar showing tonight at 7pm at the Downtown cinema ?")

In [16]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 282, 'prompt_tokens': 234, 'total_tokens': 516, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E5jxTDwuCzKHHy2G1R6e4g6Najw6a', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f9c7f-3330-7cd0-98c3-31e8df74e855-0', tool_calls=[{'name': 'check_showtimes', 'args': {'movie_title': 'Interstellar'}, 'id': 'call_6cuDeHl7bRO6ohfKvnvoZSMy', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 282, 'total_tokens': 516, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'au

<img src="../../assets/tool_runtime_information.png" width="800" height="400">
